In [2]:
import sqlite3
import json

DB_PATH = 'database/ESGdata.db'

def reconcile_data_updated(parsed_ocr, raw_id):
    """팀장님의 기존 로직을 계승하여 DB 상태까지 업데이트하는 함수"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    reporting_date = f"{parsed_ocr['year']}-{parsed_ocr['month']}"
    
    # [기존 쿼리 유지] 매핑 테이블 조인하여 실적 조회
    query = """
    SELECT s.id, map.site_id, s.value, s.metric
    FROM site_metric_map map
    JOIN standard_usage s ON map.site_id = s.site_id AND map.metric = s.metric
    WHERE map.customer_number = ? AND s.reporting_date = ?
    """
    
    cursor.execute(query, (parsed_ocr['customer_number'], reporting_date))
    db_record = cursor.fetchone()
    
    if not db_record:
        print(f"❌ [ID {raw_id}] DB 실적 미존재: 고객번호 {parsed_ocr['customer_number']}, {reporting_date}")
        conn.close()
        return

    std_id, site_id, db_value, metric = db_record
    ocr_usage = parsed_ocr['usage']
    
    # 📊 정합성 검증 로직 (Gap 분석)
    gap = db_value - ocr_usage
    abs_gap = abs(gap)
    
    # 판정 기준 (MECE 구조)
    if abs_gap < 1.0:
        final_status = 5
        diagnosis = "✅ 정합성 확인 완료 (데이터 일치)"
    elif abs(db_value * 1000 - ocr_usage) < 1.0 or abs(db_value / 1000 - ocr_usage) < 1.0:
        final_status = 4
        diagnosis = "⚠️ 단위 오류 감지 (kWh <-> MWh 등)"
    else:
        final_status = 3
        diagnosis = f"❌ 데이터 불일치 발생 (차이: {gap})"

    # [수정/추가] 결과 DB 반영
    # 1. standard_usage 상태 업데이트
    cursor.execute("UPDATE standard_usage SET v_status = ? WHERE id = ?", (final_status, std_id))
    
    # 2. 검증 로그 기록 (Audit Trail)
    cursor.execute("""
        INSERT INTO verification_logs (std_id, gap_value, result_code, diagnosis)
        VALUES (?, ?, ?, ?)
    """, (std_id, gap, final_status, diagnosis))

    # 3. OCR 원천 데이터 처리 완료 표시
    cursor.execute("UPDATE raw_ocr_data SET processing_status = 'Success' WHERE id = ?", (raw_id,))

    print(f"[{site_id}] {reporting_date} {metric} -> {diagnosis}")
    
    conn.commit()
    conn.close()

def run_batch_verification():
    """raw_ocr_data의 모든 데이터를 순차적으로 검증 실행"""
    with sqlite3.connect(DB_PATH) as conn:
        # 아직 처리되지 않은 OCR 데이터들 가져오기
        pending_data = conn.execute("SELECT id, raw_content FROM raw_ocr_data WHERE processing_status = 'Pending'").fetchall()
    
    print(f"🚀 총 {len(pending_data)}건의 증빙 대조를 시작합니다.")
    for r_id, raw_content in pending_data:
        parsed_ocr = json.loads(raw_content)
        reconcile_data_updated(parsed_ocr, r_id)
    print("🏁 검증 공정이 완료되었습니다.")

if __name__ == "__main__":
    run_batch_verification()

🚀 총 45건의 증빙 대조를 시작합니다.
[Site A] 2025-01 electricity_mwh -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-01 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-02 electricity_mwh -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-02 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-03 electricity_mwh -> ❌ 데이터 불일치 발생 (차이: -150.0)
[Site A] 2025-03 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-04 electricity_mwh -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-04 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-05 electricity_mwh -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-05 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-06 electricity_mwh -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-06 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-07 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-08 electricity_mwh -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-08 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-09 electricity_mwh -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-09 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-10 lng_nm3 -> ✅ 정합성 확인 완료 (데이터 일치)
[Site A] 2025-11 electrici